\# Computational Theory Assessment \*\*Module:\*\* Computational Theory \*\*Repository:\*\* [github.com/FionnQ-H1/comp-theory](https://github.com/FionnQ-H1/comp-theory) \*\*Author:\*\* Fionn --- ## Overview This notebook contains the implementation, theoretical analysis, and documentation for the SHA-256 algorithm assessment as specified in the NIST Secure Hash Standard (FIPS PUB 180-4). --- ## Problem 0 - Project Management &amp; Issue Tracking \* \*\*GitHub Issues:\*\* [github.com/FionnQ-H1/comp-theory/issues](https://github.com/FionnQ-H1/comp-theory/issues) Task tracking and incremental project development are recorded via GitHub Issues. --- ## Problem 1 - SHA-256 Data Representations ### Overview SHA-256 operates on specific data sizes and formats. This section explores and demonstrates how to represent: 1\. 32-bit Words (\`uint32\`) 2\. 512-bit Message Blocks 3\. Input Messages &amp; Byte Strings 4\. Big-Endian Integer Byte Ordering 5\. 256-bit Hash Outputs

### 1. Representing 32-bit Words

SHA-256 works mostly with **32-bit unsigned integers**, known as words. Each word holds a value from 0 to \\(2^{32}-1\\) (0 to 4,294,967,295), as set out in the [NIST FIPS PUB 180-4 Standard](https://doi.org/10.6028/NIST.FIPS.180-4).

Python's built-in `int` has no fixed size. It keeps growing to fit whatever number it's given. That doesn't suit SHA-256, which needs every word to stay exactly 32 bits wide, with addition wrapping back round once it passes \\(2^{32}\\) (addition modulo \\(2^{32}\\)).

To get that behaviour, we use [`numpy.uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint32). It fixes each value at 32 bits and wraps automatically on overflow, the same way hardware does.

import numpy as np 
# Define maximum 32-bit word value (2^32 - 1) 
max\_word = np.uint32(0xFFFFFFFF) # 4,294,967,295 in decimal 
# Standard Python integer addition (does NOT wrap, grows to 33 bits) 
python\_add = int(max\_word) + 1 
# NumPy uint32 addition (wraps around modulo 2^32 back to 0) 
numpy\_add = max\_word + np.uint32(1) 
print(f"Max 32-bit Word (Hex) : {hex(max\_word)}") 
print(f"Max 32-bit Word (Decimal): {max\_word}") 
print(f"Standard Python Addition : {python\_add} (33 bits)") 
print(f"NumPy uint32 Addition : {numpy\_add} (Wrapped to 0)")

### 2. Sequences of 32-bit Words

SHA-256 structures data into fixed-length sequences of 32-bit words throughout its processing pipeline:
* **Message Blocks (\\(M\\)):** Each 512-bit block is parsed into a sequence of **16 words** (\\(16 \times 32 = 512\\) bits).
* **Message Schedule (\\(W\\)):** The 16-word block is expanded into a **64-word sequence** (\\(W_0 \dots W_{63}\\)).
* **Hash State (\\(H\\)):** Initial values, working variables (\\(a \dots h\\)), and final digests are stored as sequences of **8 words** (\\(8 \times 32 = 256\\) bits).

To store and manipulate these word sequences efficiently while maintaining strict 32-bit types across all elements, we use 1D [NumPy arrays](https://numpy.org/doc/stable/reference/generated/numpy.array.html) with `dtype=np.uint32`.

In [3]:
import numpy as np

# Creates a sequence of 16 32 bit words representing a 512 bit block
words_sequence = np.array([
    0x61626364, 0x80000000, 0x00000000, 0x00000000, 
    0x00000000, 0x00000000, 0x00000000, 0x00000000, 
    0x00000000, 0x00000000, 0x00000000, 0x00000000, 
    0x00000000, 0x00000000, 0x00000000, 0x00000018
], dtype=np.uint32)

print(f"Number of words in sequence : {len(words_sequence)}") 
print(f"Data type of array elements : {words_sequence.dtype}") 
print(f"Total size in bytes : {words_sequence.nbytes} bytes ({words_sequence.nbytes * 8} bits)")
print(f"First word (W_0 in hex) : {hex(words_sequence[0])}")

Number of words in sequence : 16
Data type of array elements : uint32
Total size in bytes : 64 bytes (512 bits)
First word (W_0 in hex) : 0x61626364
